In [ ]:
def run_sa_trial_with_trace(Gamma, A, N, beta, propose_move, laydowncoordinates,
                             valid_fold, energy, H, log_every=100):
    while True:
        d = np.random.randint(0, 4, size=N - 1)
        coordinates = laydowncoordinates(d)
        if valid_fold(coordinates):
            break

    E = energy(coordinates, Gamma, A, H)
    best_E = E
    trace = [E]

    for i in range(len(beta)):
        while True:
            d_new = propose_move(d)
            coords_new = laydowncoordinates(d_new)
            if valid_fold(coords_new):
                break
        E_new = energy(coords_new, Gamma, A, H)
        deltaE = E_new - E

        p_accept = 1.0 / (1.0 + np.exp(deltaE * beta[i]))
        if np.random.rand() < p_accept:
            d, coordinates, E = d_new, coords_new, E_new
            if E < best_E:
                best_E = E

        if (i + 1) % log_every == 0:
            trace.append(E)

    return best_E, E, np.array(trace)

# run a small batch per schedule (classical + tuned + your beta_of_t sigmoid, reconstructed
# from params saved in reinforce_sigmoid_schedule_cpu.pkl — no need to touch the other server)
def beta_of_t(t, beta_min, beta_max, k, t0):
    sigma_t = 1.0 / (1.0 + np.exp(-k * (t - t0)))
    return beta_min + (beta_max - beta_min) * sigma_t

t_arr = np.arange(num_sweeps)
reinforce_beta = beta_of_t(t_arr, *rf["best_params"])   # rf = your loaded pkl from before

n_conv_trials = 100
log_every = 100

traces = {}
for name, beta in {**{n: np.linspace(0.1, 10, num_sweeps) for n in classical_names},
                    "Linear (tuned)": beta_tuned,
                    "REINFORCE (sigmoid)": reinforce_beta}.items():
    runs = [run_sa_trial_with_trace(Gamma, A, N, beta, propose_move,
                                     laydowncoordinates, valid_fold, energy, H,
                                     log_every=log_every)[2]
            for _ in range(n_conv_trials)]
    traces[name] = np.array(runs)   # shape (n_conv_trials, n_checkpoints)

# plot mean energy vs sweep, with bootstrap-style band (just std here for speed)
sweep_axis = np.arange(0, num_sweeps + 1, log_every)
plt.figure(figsize=(9, 5))
for name, arr in traces.items():
    mean_trace = arr.mean(axis=0)
    std_trace = arr.std(axis=0)
    plt.plot(sweep_axis, mean_trace, label=name)
    plt.fill_between(sweep_axis, mean_trace - std_trace, mean_trace + std_trace, alpha=0.15)
plt.xlabel("Sweep")
plt.ylabel("Energy")
plt.title(f"Convergence Speed by Schedule (n={n_conv_trials} trials each)")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.savefig("convergence_speed.png", dpi=150)
plt.show()